In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
import polars as pl
import polars.selectors as cs
import lightgbm as lgb
from sklearn.inspection import permutation_importance

In [2]:
X_train = pl.read_csv('data/train.csv')
X_test = pl.read_csv('data/test.csv')
y_train = (
    X_train.get_column('Churn')
    .replace({"Yes": 1, "No": 0})
    .cast(pl.Int8)
)
X_train = X_train.drop('Churn')

In [3]:
X_train = X_train.drop('id')
X_test = X_test.drop('id')

In [4]:
X_cv = X_train.with_columns(
    tenure_safe = pl.when(pl.col('tenure') == 0).then(0.001).otherwise(pl.col('tenure'))
).with_columns(
    Int_PriceShockRatio = pl.col('MonthlyCharges') / ((pl.col('TotalCharges') / pl.col('tenure_safe')) + 0.001),
    Int_CostPerTenure = pl.col('MonthlyCharges') / pl.col('tenure_safe'),
    Int_ImpliedTenureDiff = (pl.col('TotalCharges') / (pl.col('MonthlyCharges') + 0.001)) - pl.col('tenure'),
    Int_CumulativeBurden = pl.col('TotalCharges') * pl.col('MonthlyCharges')
)

In [5]:
X_test = X_test.with_columns(
    tenure_safe = pl.when(pl.col('tenure') == 0).then(0.001).otherwise(pl.col('tenure'))
).with_columns(
    Int_PriceShockRatio = pl.col('MonthlyCharges') / ((pl.col('TotalCharges') / pl.col('tenure_safe')) + 0.001),
    Int_CostPerTenure = pl.col('MonthlyCharges') / pl.col('tenure_safe'),
    Int_ImpliedTenureDiff = (pl.col('TotalCharges') / (pl.col('MonthlyCharges') + 0.001)) - pl.col('tenure'),
    Int_CumulativeBurden = pl.col('TotalCharges') * pl.col('MonthlyCharges')
)

In [6]:
family_cols = ['Partner', 'Dependents']
X_cv = X_cv.with_columns(
    FamilySize = pl.sum_horizontal(pl.col(family_cols) == 'Yes')
)
X_test = X_test.with_columns(
    FamilySize = pl.sum_horizontal(pl.col(family_cols) == 'Yes')
)

In [7]:
SERVICE_COLS = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

X_cv = X_cv.with_columns(
    service_count = pl.sum_horizontal(pl.col(SERVICE_COLS) == 'Yes').cast(pl.Float32)
)
X_test = X_test.with_columns(
    service_count = pl.sum_horizontal(pl.col(SERVICE_COLS) == 'Yes').cast(pl.Float32)
)


In [8]:
tree_binner = DecisionTreeClassifier(max_depth=2, random_state=42)

In [9]:
cols_to_bin = ['tenure', 'MonthlyCharges', 'TotalCharges']
cut_exprs = [] 

for col in cols_to_bin:
    temp_X = X_cv.select(col).to_numpy()
    
    tree_binner.fit(temp_X, y_train)

    thresholds = tree_binner.tree_.threshold
    splits = [th for th in thresholds if th != -2.0]
    splits = sorted(list(np.unique(splits)))
    
    opt_przedzialy = [-np.inf] + splits + [np.inf]
    print(opt_przedzialy, col)

    opt_etykiety = [f"{col}_Bin_{i+1}" for i in range(len(splits) + 1)]
    
    bin_col_name = f"{col}_binned"

    expr = (
        pl.col(col)
        .cut(breaks=splits, labels=opt_etykiety)
        .cast(pl.String) 
        .alias(bin_col_name) 
    )
    
    cut_exprs.append(expr)

X_cv = X_cv.with_columns(cut_exprs)
X_test = X_test.with_columns(cut_exprs)

[-inf, np.float64(1.5), np.float64(17.5), np.float64(43.5), inf] tenure
[-inf, np.float64(26.824999809265137), np.float64(68.9749984741211), np.float64(106.9749984741211), inf] MonthlyCharges
[-inf, np.float64(198.0500030517578), np.float64(920.2250061035156), np.float64(4283.5), inf] TotalCharges


In [10]:
X_cv = X_cv.with_columns([
    (pl.col("MonthlyCharges") / (pl.col("service_count") + 1)).alias("AvgCostPerService")
])
X_test = X_test.with_columns([
    (pl.col("MonthlyCharges") / (pl.col("service_count") + 1)).alias("AvgCostPerService")
])

print(X_cv.select(["service_count", "AvgCostPerService", "MonthlyCharges"]).head(5))

shape: (5, 3)
┌───────────────┬───────────────────┬────────────────┐
│ service_count ┆ AvgCostPerService ┆ MonthlyCharges │
│ ---           ┆ ---               ┆ ---            │
│ f32           ┆ f64               ┆ f64            │
╞═══════════════╪═══════════════════╪════════════════╡
│ 4.0           ┆ 12.02             ┆ 60.1           │
│ 5.0           ┆ 11.583333         ┆ 69.5           │
│ 5.0           ┆ 16.733333         ┆ 100.4          │
│ 1.0           ┆ 34.85             ┆ 69.7           │
│ 1.0           ┆ 35.225            ┆ 70.45          │
└───────────────┴───────────────────┴────────────────┘


In [11]:
X_cv = X_cv.with_columns([
    pl.col("MonthlyCharges").mean().over("Contract").alias("AvgMonthly_ByContract"),
    pl.col("MonthlyCharges").mean().over("InternetService").alias("AvgMonthly_ByInternet"),
])
X_cv = X_cv.with_columns([
    (pl.col("MonthlyCharges") / pl.col("AvgMonthly_ByContract")).alias("ChargeToContractAvg_Ratio"),
    (pl.col("MonthlyCharges") / pl.col("AvgMonthly_ByInternet")).alias("ChargeToInternetAvg_Ratio")
])

X_test = X_test.with_columns([
    pl.col("MonthlyCharges").mean().over("Contract").alias("AvgMonthly_ByContract"),
    pl.col("MonthlyCharges").mean().over("InternetService").alias("AvgMonthly_ByInternet"),
])
X_test = X_test.with_columns([
    (pl.col("MonthlyCharges") / pl.col("AvgMonthly_ByContract")).alias("ChargeToContractAvg_Ratio"),
    (pl.col("MonthlyCharges") / pl.col("AvgMonthly_ByInternet")).alias("ChargeToInternetAvg_Ratio")
])

In [12]:
cols_for_te = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaymentMethod', 'tenure_binned', 'MonthlyCharges_binned', 
    'TotalCharges_binned', 'FamilySize'
]

In [13]:
X_cv = X_cv.with_columns(y_train.alias("target_tmp"))

In [14]:
encoder = TargetEncoder(target_type="binary", cv=5, smooth="auto")
train_encoded_numpy = encoder.fit_transform(
    X_cv.select(cols_for_te).to_pandas(), 
    y_train.to_numpy()
)

test_encoded_numpy = encoder.transform(
    X_test.select(cols_for_te).to_pandas()
)

encoded_col_names = [f"{col}_TE" for col in cols_for_te]

X_cv = pl.concat([
    X_cv.drop(cols_for_te),
    pl.DataFrame(train_encoded_numpy, schema=encoded_col_names)
], how="horizontal")

X_test = pl.concat([
    X_test.drop(cols_for_te),
    pl.DataFrame(test_encoded_numpy, schema=encoded_col_names)
], how="horizontal")

In [15]:
X_cv = X_cv.with_columns(pl.col('PaperlessBilling').replace({'Yes': 1, 'No': 0}))
X_test = X_test.with_columns(pl.col('PaperlessBilling').replace({'Yes': 1, 'No': 0}))

In [16]:
val_loyalty_features = ["tenure", "MonthlyCharges", "TotalCharges"]

data_loyalty_cv = X_cv.select(val_loyalty_features).to_numpy()
data_loyalty_test = X_test.select(val_loyalty_features).to_numpy()

scaler_loyalty = StandardScaler()
data_loyalty_cv_scaled = scaler_loyalty.fit_transform(data_loyalty_cv)
data_loyalty_test_scaled = scaler_loyalty.transform(data_loyalty_test)
kmeans_val = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_cv = kmeans_val.fit_predict(data_loyalty_cv_scaled)
clusters_test = kmeans_val.predict(data_loyalty_test_scaled)
distances_cv = kmeans_val.transform(data_loyalty_cv_scaled)
distances_test = kmeans_val.transform(data_loyalty_test_scaled)

dist_cols = [f"dist_val_loyalty_c{i}" for i in range(distances_cv.shape[1])]
dist_df_cv = pl.DataFrame(distances_cv, schema=dist_cols)
dist_df_test = pl.DataFrame(distances_test, schema=dist_cols)

X_cv = pl.concat([X_cv, dist_df_cv], how="horizontal").with_columns(
    pl.Series("cluster_val_loyalty", clusters_cv).cast(pl.Utf8)
)

X_test = pl.concat([X_test, dist_df_test], how="horizontal").with_columns(
    pl.Series("cluster_val_loyalty", clusters_test).cast(pl.Utf8)
)


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Nie można odnaleźć określonego pliku
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python313\

In [17]:
behavioral_features = [
    "Contract_TE", "PaymentMethod_TE", "OnlineSecurity_TE", 
    "TechSupport_TE", "InternetService_TE", "PaperlessBilling"
]

In [18]:
data_beh_cv = X_cv.select(behavioral_features).to_numpy()
data_beh_test = X_test.select(behavioral_features).to_numpy()

scaler_beh = StandardScaler()
data_beh_cv_scaled = scaler_beh.fit_transform(data_beh_cv)
data_beh_test_scaled = scaler_beh.transform(data_beh_test)

kmeans_beh = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_beh_cv = kmeans_beh.fit_predict(data_beh_cv_scaled)
clusters_beh_test = kmeans_beh.predict(data_beh_test_scaled)

X_cv = X_cv.with_columns(
    pl.Series("cluster_val_behavioral", clusters_beh_cv).cast(pl.Utf8)
)

X_test = X_test.with_columns(
    pl.Series("cluster_val_behavioral", clusters_beh_test).cast(pl.Utf8)
)

In [19]:
to_drop = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService']
X_cv = X_cv.drop(to_drop)
X_test = X_test.drop(to_drop)

In [20]:
X_cv_pd = X_cv.to_pandas()
X_test_pd = X_test.to_pandas()

In [21]:
categorical_cols = ['cluster_val_loyalty', 'cluster_val_behavioral']
y_train_np = np.array(y_train).ravel()

encoder = TargetEncoder(target_type="binary", cv=5, smooth="auto", random_state=42)

train_encoded_numpy = encoder.fit_transform(
    X_cv_pd[categorical_cols], 
    y_train_np
)

test_encoded_numpy = encoder.transform(
    X_test_pd[categorical_cols]
)

encoded_col_names = [f"{col}_TE" for col in categorical_cols]

X_cv_pd = pd.concat([
    X_cv_pd.drop(columns=categorical_cols),
    pd.DataFrame(train_encoded_numpy, columns=encoded_col_names, index=X_cv_pd.index)
], axis=1)

X_test_pd = pd.concat([
    X_test_pd.drop(columns=categorical_cols),
    pd.DataFrame(test_encoded_numpy, columns=encoded_col_names, index=X_test_pd.index)
], axis=1)

In [22]:
X_cv_pd['PaperlessBilling'] = X_cv_pd['PaperlessBilling'].astype(int)

X_test_pd['PaperlessBilling'] = X_test_pd['PaperlessBilling'].astype(int)

print(f"Typ PaperlessBilling w CV: {X_cv_pd['PaperlessBilling'].dtype}")

Typ PaperlessBilling w CV: int64


In [23]:
model = lgb.LGBMClassifier(
    n_estimators=50, 
    random_state=42, 
    n_jobs=-1
)

model.fit(X_cv_pd, y_train)
print("Model trained successfully.")
result = permutation_importance(
    model, 
    X_cv_pd, 
    y_train, 
    n_repeats=5, 
    random_state=42, 
    n_jobs=-1
)

importance_df = pd.DataFrame({
    'Feature': X_cv_pd.columns,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std
}).sort_values(by='Importance_Mean', ascending=False)

print(importance_df)

[LightGBM] [Info] Number of positive: 133817, number of negative: 460377
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052128 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3768
[LightGBM] [Info] Number of data points in the train set: 594194, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235573
[LightGBM] [Info] Start training from score -1.235573
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.